In [1]:
#Importing the standard libraries
import numpy as np
import matplotlib.pyplot as plt
%matplotlib qt

#Importing the solver modules
import system
from obe import obe 
from states import SigmaLevel,PiLevelParity
from obe import Excitation
import time
import warnings
warnings.filterwarnings('ignore')
import datetime

/Users/mangesh/Library/CloudStorage/Box-Box/Density_Matrix/BaF138/sidebands/N_0_137/sinusoidal_driving/experiment_with_cython/obe.py:28: UserWarning: Could not detect diffeqpy. Only Python package for OBE solver available.
  warnings.warn(f"Could not detect diffeqpy. Only Python package for OBE solver available.")


from field_optimization import *
Bz_list = np.arange(0,60,5)+0.01
rabi = 15.0
detuning = 0
with open("field_opt_results.txt", "w") as f:
    for Bz in Bz_list:
        sum_z,sum_x,desired_Z,desired_X = field_optimizer(Bz,rabi,detuning,level_to_optimize = 15)
        print(np.round(Bz,2),np.round(sum_z,3),np.round(sum_x,3),np.round(sum_z+sum_x,3))
        f.write(f"{np.round(Bz,2)}, {np.round(sum_z,3)}, {np.round(sum_x,3)}, {np.round(sum_z+sum_x,3)}\n")
        f.write(f"desired_Z : {desired_Z}\n")
        f.write(f"desired_X : {desired_X}\n")    

print(desired_Z)
print(desired_X)

In [2]:
Bz = 20.0
b=system.System([0,2],['1/2-','3/2-','5/2-'],B_field = [0.0,0.0,Bz],ignore_mF = False)

start = time.time()

b.sigma_Hamiltonian.generate_bare()
b.sigma_Hamiltonian.Zeeman.generate_Zeeman()
b.pi_Hamiltonian.generate_bare()
b.pi_Hamiltonian.Zeeman.generate_Zeeman()

#Next diagonalize the Hamiltonian for this system
b.sigma_Hamiltonian.diagonalize()
b.pi_Hamiltonian.diagonalize()


G_global = b.sigma_Hamiltonian.diagonalized_states
GH_global = np.round(b.sigma_Hamiltonian.diagonalized_Hamiltonian,3)
E_global =b.pi_Hamiltonian.diagonalized_states
EH_global = np.round(b.pi_Hamiltonian.diagonalized_Hamiltonian,3)


print(f"Static Hamiltonian took : {time.time()-start}s. ")

G = G_global
GH = GH_global
E = E_global[0:16]
EH = EH_global[0:16,0:16]
b.generate_branching_ratios(G,E)
BR = b.branching_ratios

# Edit entries in GH and EH

GH -= np.amin(np.diag(GH))*np.identity(np.shape(GH)[0])
EH -= np.amin(np.diag(EH))*np.identity(np.shape(EH)[0])

G0_mean = np.mean(np.diag(GH)[0:16])
G2_mean = np.mean(np.diag(GH)[16:80+16])
E_mean =  np.mean(np.diag(EH))



Static Hamiltonian took : 0.9373860359191895s. 
Pi branching took : 1.6577069759368896 sec
Sigma+ branching took : 0.4206709861755371 sec
Sigma- branching took : 0.40508103370666504 sec


In [3]:
G[0]

(-0.99995+0j) |G = 1.0, N = 0, F1 = 1.0, F = 1.5, mF = 1.5> + 
(-0.00912+0j) |G = 2.0, N = 0, F1 = 2.0, F = 1.5, mF = 1.5> + 
(0.00457+0j) |G = 2.0, N = 0, F1 = 2.0, F = 2.5, mF = 1.5> + 
(-0.00089+0j) |G = 1.0, N = 2, F1 = 1.0, F = 1.5, mF = 1.5> + 
(3e-05+0j) |G = 1.0, N = 2, F1 = 2.0, F = 1.5, mF = 1.5> + 
(-1e-05+0j) |G = 1.0, N = 2, F1 = 2.0, F = 2.5, mF = 1.5> + 
(-0.00122+0j) |G = 2.0, N = 2, F1 = 1.0, F = 1.5, mF = 1.5> + 
(2e-05+0j) |G = 2.0, N = 2, F1 = 2.0, F = 1.5, mF = 1.5>

In [4]:
E_mean

np.complex128(389.66475000604987+0j)

In [5]:
EH[15,15]-EH[0,0]

np.complex128(630.4010000228882+0j)

In [6]:
G0_mean-np.mean(np.diag(GH)[0:6])

np.complex128(2910.8940625000005+0j)

In [7]:
GH[0,0]-GH[15,15]

np.complex128(-4723.795+0j)

In [8]:
630-390

240

In [9]:
G[0]

(-0.99995+0j) |G = 1.0, N = 0, F1 = 1.0, F = 1.5, mF = 1.5> + 
(-0.00912+0j) |G = 2.0, N = 0, F1 = 2.0, F = 1.5, mF = 1.5> + 
(0.00457+0j) |G = 2.0, N = 0, F1 = 2.0, F = 2.5, mF = 1.5> + 
(-0.00089+0j) |G = 1.0, N = 2, F1 = 1.0, F = 1.5, mF = 1.5> + 
(3e-05+0j) |G = 1.0, N = 2, F1 = 2.0, F = 1.5, mF = 1.5> + 
(-1e-05+0j) |G = 1.0, N = 2, F1 = 2.0, F = 2.5, mF = 1.5> + 
(-0.00122+0j) |G = 2.0, N = 2, F1 = 1.0, F = 1.5, mF = 1.5> + 
(2e-05+0j) |G = 2.0, N = 2, F1 = 2.0, F = 1.5, mF = 1.5>

In [10]:
G[0].states[0].N

0

import cProfile
import pstats

with cProfile.Profile() as pr:
    b.generate_branching_ratios(G,E)

stats = pstats.Stats(pr)
stats.sort_stats("tottime").print_stats(20)

In [11]:
H0 = np.zeros((len(G)+len(E),len(G)+len(E)),dtype=np.complex128)
H0[0:len(G),0:len(G)] = GH
H0[len(G):,len(G):] = EH
b.generate_interaction_Hamiltonian(G,E)
Hint_1=np.round(b.interaction_Hamiltonian,3)
n0 = [1/len(G)]*len(G)+[0]*len(E)

b.generate_interaction_Hamiltonian(G,E,pol=+1)
Hint_2 = np.round(b.interaction_Hamiltonian,3)

b.generate_interaction_Hamiltonian(G,E,pol=-1)
Hint_3 = np.round(b.interaction_Hamiltonian,3)

In [12]:
level_to_optimize = 15
myList_1 = []
myList_2 = []
for i in range(0,len(G)):
    if i != level_to_optimize:
        tempList_1 = []
        tempList_2 = []
        for j in range(len(E)):
            #look at the Hint element for z pol light
            if np.abs(Hint_1[i,len(G)+j])> 1e-3:
                tempList_1.append((i,j))
            if np.abs(Hint_2[i,len(G)+j])> 1e-3 or np.abs(Hint_3[i,len(G)+j])> 1e-3:
                tempList_2.append((i,j))
            
        myList_1.append(tempList_1)
        myList_2.append(tempList_2)

In [13]:
#to select a random neighbour
Gamma = 2*np.pi*2.7
tsigma = 8.192/4
import random
def randomize_transition(myList,polarization):
    """ returns a list where each element is  tuple of ground and excited state index for the transition"""
    candidate_fields = []
    for item in myList:
        if len(item) != 0:
            candidate_fields.append(random.choice(item))
    #print(candidate_fields)
    field = []
    for item in candidate_fields:
        if len(item) != 0:
            (i,j) = item
        else:
            continue
        rabi = random.uniform(0.1, 5.0)*Gamma
        pol = polarization
        groundState = G[i]
        excitedState = E[j]
        det = random.uniform(-1,1)*Gamma
        pos = 0
        dia = 4*tsigma
        temp_field = Excitation(rabi,pol,groundState,excitedState,detuning = det,position = pos,diameter = dia ,shape = "Uniform")
        field.append(temp_field)
    
    return field,candidate_fields


In [14]:
#to select a desired set of transitions
Gamma = 2*np.pi*2.7
tsigma = 8.192/4
import random
def generate_transition(desired_transitions,polarization):
    
    field = []
    for item in desired_transitions:
        (i,j) = item
        rabi = random.uniform(0.1, 5.0)*Gamma
        pol = polarization
        groundState = G[i]
        excitedState = E[j]
        det = random.uniform(-1,1)*Gamma
        pos = 0
        dia = 4*tsigma
        temp_field = Excitation(rabi,pol,groundState,excitedState,detuning = det,position = pos,diameter = dia ,shape = "Uniform")
        field.append(temp_field)
    
    return field,desired_transitions

In [15]:
def flatten(x):
    temp_list = []
    for item in x:
        if type(item) == list:
            for temp_item in item:
                temp_list.append(temp_item)
        else:
            temp_list.append(item)
    return temp_list


In [16]:
import copy
current_field,transitions_Z = randomize_transition(myList_1,0)
N_z= len(current_field)
temp_field,transitions_sigma = randomize_transition(myList_2,1)
current_field.append(temp_field)
N_sigma = len(current_field[-1])
current_field=flatten(current_field)

In [17]:
### only when we want to force a choice of transition

desired_Z = [(0, 0), (1, 1), (2, 2), (3, 3), (4, 4), (5, 5), (6, 6), (7, 2), (8, 1), (9, 0), (10, 10), (11, 3), (12, 2), (13, 1), (14, 0), (17, 15), (18, 14), (19, 13), (20, 12), (21, 11), (22, 10), (24, 15), (25, 14), (26, 13), (27, 2), (28, 3), (29, 10), (30, 15), (31, 0), (32, 1), (33, 2), (34, 0), (35, 3), (36, 10), (37, 1), (38, 2), (39, 3), (40, 7), (41, 1), (42, 3), (43, 2), (44, 1), (45, 0), (46, 1), (47, 7), (48, 3), (49, 2), (50, 1), (51, 0), (52, 2), (53, 4), (54, 6), (55, 2), (56, 1), (57, 10), (58, 0), (59, 3), (60, 2), (61, 1), (62, 0), (63, 15), (64, 10), (65, 6), (66, 12), (67, 13), (68, 14), (69, 15), (71, 10), (72, 11), (73, 12), (74, 13), (75, 14), (76, 15), (79, 10), (80, 11), (81, 12), (82, 13), (83, 14), (84, 15), (88, 10), (89, 11), (90, 12), (91, 13), (92, 14), (93, 15)]
desired_X = [(0, 13), (1, 14), (2, 13), (3, 12), (4, 0), (5, 3), (6, 7), (7, 1), (8, 0), (9, 15), (10, 11), (11, 12), (12, 4), (13, 5), (14, 4), (16, 15), (17, 14), (18, 15), (19, 14), (20, 13), (21, 12), (22, 11), (23, 10), (24, 14), (25, 15), (26, 14), (27, 3), (28, 12), (29, 11), (30, 0), (31, 1), (32, 0), (33, 3), (34, 15), (35, 2), (36, 3), (37, 2), (38, 3), (39, 2), (40, 1), (41, 0), (42, 5), (43, 1), (44, 0), (45, 1), (46, 0), (47, 3), (48, 2), (49, 1), (50, 0), (51, 1), (52, 3), (53, 0), (54, 7), (55, 1), (56, 0), (57, 3), (58, 1), (59, 2), (60, 1), (61, 0), (62, 1), (63, 0), (64, 6), (65, 10), (66, 11), (67, 12), (68, 15), (69, 0), (70, 10), (71, 11), (72, 10), (73, 11), (74, 12), (75, 13), (76, 14), (77, 15), (78, 10), (79, 6), (80, 7), (81, 11), (82, 12), (83, 13), (84, 14), (85, 15), (87, 10), (88, 11), (89, 10), (90, 11), (91, 12), (92, 13), (93, 14), (94, 15)]
current_field,transitions_Z = generate_transition(desired_Z,0)
N_z= len(current_field)
temp_field,transitions_X = generate_transition(desired_X,1) 
current_field.append(temp_field)
N_sigma = len(current_field[-1])
current_field=flatten(current_field)


In [18]:
G.index(current_field[23].ground_state)

26

In [19]:
E.index(current_field[26].excited_state)

10

In [20]:
current_field[0].detuning

0.9362207642614083

In [21]:
#Count the number of transitions from the different N 
# We are going to address both G levels within the same N with the output from same laser.
Nz_0,Nz_2 = 0,0
for item in transitions_Z:
    g = item[0]                 #index of the ground state
    
    max_N_idx = np.argmax(np.abs(G[g].amplitude))
    N_g = G[g].states[max_N_idx].N

    if N_g == 0:
        Nz_0 += 1
    elif N_g == 2:
        Nz_2 += 1
    else:
        warnings.warn('Warning Message: State with N = 0 or N = 2 expected.') 
print(Nz_0,Nz_2)


Nx_0,Nx_2 = 0,0
for item in transitions_X:
    g = item[0]                 #index of the ground state
    
    max_N_idx = np.argmax(np.abs(G[g].amplitude))
    N_g = G[g].states[max_N_idx].N

    if N_g == 0:
        Nx_0 += 1
    elif N_g == 2:
        Nx_2 += 1
    else:
        warnings.warn('Warning Message: State with N = 0 or N = 2 expected.') 
print(Nx_0,Nx_2)


Ns = [Nz_0,Nz_2,
        Nx_0,Nx_2]


15 70
15 78


In [22]:
# Temporary function to get the position of the beaM GIVEN THE SEQUENCE AND DIAMETER OF THE BEAM
def get_pos(A):
    cumsum_arr = np.cumsum(A)
    pos = [A[0]/2]
    for i in range(1,len(A)):
        temp_pos = np.round((cumsum_arr[i]+cumsum_arr[i-1])*0.5,2)
        pos.append(temp_pos)
    return pos

In [23]:
#The cost function. Returns the population of the desired state.
def r22(rabi_det_params:list,Ns:list,Omegas:tuple,passes=1):
    try:
        Omega_z_0,Omega_z_2, z_x_factor = Omegas
        Omega_x_0,Omega_x_2 = z_x_factor*Omega_z_0,z_x_factor*Omega_z_2
        package = 'Python'
        obe_mode = 'symengine'
        [Nz_0,Nz_2,Nx_0,Nx_2] = Ns
        
        P_z_0 = []
        P_z_2 = []
        P_x_0 = []
        P_x_2 = []

        for i in range(len(current_field)): #Converting between Nevergrad array and field rabis and detunings
                current_field[i].rabi = rabi_det_params[i][0]
                current_field[i].detuning = rabi_det_params[i][1]
                if i < Nz_0:
                    P_z_0.append(rabi_det_params[i][0]**2)
                elif i >= Nz_0 and i < (Nz_0 + Nz_2):
                    P_z_2.append(rabi_det_params[i][0]**2)
                elif i >= (Nz_0 + Nz_2) and i < (Nz_0 + Nz_2 + Nx_0):
                    P_x_0.append(rabi_det_params[i][0]**2)
                else:
                    P_x_2.append(rabi_det_params[i][0]**2)
        
        total_pow_z_0 = np.sum(P_z_0)
        total_pow_z_2 = np.sum(P_z_2)
        total_pow_x_0 = np.sum(P_x_0)
        total_pow_x_2 = np.sum(P_x_2)
        
        
        P_z_0 = np.array(P_z_0)/np.sum(P_z_0)
        P_z_2 = np.array(P_z_2)/np.sum(P_z_2)
        P_x_0 = np.array(P_x_0)/np.sum(P_x_0)
        P_x_2 = np.array(P_x_2)/np.sum(P_x_2)
        
        
        threshold = 0.01 #rise time of the pulse signal used
        P_z_0[P_z_0 < threshold] = 0
        P_z_2[P_z_2 < threshold] = 0
        P_x_0[P_x_0 < threshold] = 0
        P_x_2[P_x_2 < threshold] = 0

        total_pow_z_0 = np.sum(P_z_0)
        total_pow_z_2 = np.sum(P_z_2)
        total_pow_x_0 = np.sum(P_x_0)
        total_pow_x_2 = np.sum(P_x_2)        
        
        P_z_0 = np.array(P_z_0)/total_pow_z_0*4*tsigma
        P_z_2 = np.array(P_z_2)/total_pow_z_2*4*tsigma
        P_x_0 = np.array(P_x_0)/total_pow_x_0*4*tsigma
        P_x_2 = np.array(P_x_2)/total_pow_x_2*4*tsigma        

        pos_z_0 = np.round(get_pos(P_z_0),5)
        pos_z_2 = np.round(get_pos(P_z_2),5)
        pos_x_0 = np.round(get_pos(P_x_0),5)
        pos_x_2 = np.round(get_pos(P_x_2),5)


        field = current_field
        field_Z_0 = field[0:
                            Nz_0]
        field_Z_2 = field[Nz_0:
                                Nz_0+Nz_2]
        field_X_0 = field[Nz_0+Nz_2:
                                    Nz_0+Nz_2+Nx_0]
        field_X_2 = field[Nz_0+Nz_2+Nx_0:
                                        Nz_0+Nz_2+Nx_0+Nx_2]
    
        #Modify fields for N = 0 
        ############################
        for i in range(len(field_Z_0)):
            field_Z_0[i].position = pos_z_0[i]
            field_Z_0[i].diameter = P_z_0[i]
            field_Z_0[i].rabi = Omega_z_0

        Z0_field = []
        for item in field_Z_0:
            Z0_field.append(item)
            item_other_sideband = copy.copy(item)
            #determine frequency to shift by
            #What is the frequency of this light 
            idx_ground,idx_excited = G.index(item.ground_state), E.index(item.excited_state)
            freq_light = EH[idx_excited,idx_excited]-GH[idx_ground,idx_ground] + item.detuning
            delta = freq_light - (E_mean - G0_mean)
            new_det = item.detuning-2*delta
            item_other_sideband.detuning = new_det
            Z0_field.append(item_other_sideband)

        for i in range(len(field_X_0)):
            field_X_0[i].position = pos_x_0[i]
            field_X_0[i].diameter = P_x_0[i]
            field_X_0[i].rabi = 1/np.sqrt(2)*Omega_x_0

        X0_field = []
        for item in field_X_0:
            X0_field.append(item)
            
            #pol reversed item
            item_pol_reversed = copy.copy(item)
            item_pol_reversed.pol *= -1
            X0_field.append(item_pol_reversed)

            #other sideband
            item_other_sideband = copy.copy(item)
            idx_ground,idx_excited = G.index(item.ground_state), E.index(item.excited_state)
            freq_light = EH[idx_excited,idx_excited]-GH[idx_ground,idx_ground] + item.detuning
            delta = freq_light - (E_mean - G0_mean)
            new_det = item.detuning-2*delta
            item_other_sideband.detuning = new_det
            X0_field.append(item_other_sideband)

            #other sideband pol reversed
            item_other_sideband_pol_reversed = copy.copy(item_other_sideband)
            item_other_sideband_pol_reversed.pol *= -1
            X0_field.append(item_other_sideband_pol_reversed)



        #Modify fields for N = 2
        ######################### 
        for i in range(len(field_Z_2)):
            field_Z_2[i].position = pos_z_2[i]
            field_Z_2[i].diameter = P_z_2[i]
            field_Z_2[i].rabi = Omega_z_2

        Z2_field = []
        for item in field_Z_2:
            Z2_field.append(item)
            item_other_sideband = copy.copy(item)
            #determine frequency to shift by
            #What is the frequency of this light 
            idx_ground,idx_excited = G.index(item.ground_state), E.index(item.excited_state)
            freq_light = EH[idx_excited,idx_excited]-GH[idx_ground,idx_ground] + item.detuning
            delta = freq_light - (E_mean - G2_mean)
            new_det = item.detuning-2*delta
            item_other_sideband.detuning = new_det
            Z2_field.append(item_other_sideband)

        for i in range(len(field_X_2)):
            field_X_2[i].position = pos_x_2[i]
            field_X_2[i].diameter = P_x_2[i]
            field_X_2[i].rabi = 1/np.sqrt(2)*Omega_x_2

        X2_field = []
        for item in field_X_2:
            X2_field.append(item)
            
            #pol reversed item
            item_pol_reversed = copy.copy(item)
            item_pol_reversed.pol *= -1
            X2_field.append(item_pol_reversed)

            #other sideband
            item_other_sideband = copy.copy(item)
            idx_ground,idx_excited = G.index(item.ground_state), E.index(item.excited_state)
            freq_light = EH[idx_excited,idx_excited]-GH[idx_ground,idx_ground] + item.detuning
            delta = freq_light - (E_mean - G2_mean)
            new_det = item.detuning-2*delta
            item_other_sideband.detuning = new_det
            X2_field.append(item_other_sideband)

            #other sideband pol reversed
            item_other_sideband_pol_reversed = copy.copy(item_other_sideband)
            item_other_sideband_pol_reversed.pol *= -1
            X2_field.append(item_other_sideband_pol_reversed)
        

        steps=5000
        n=len(E)+len(G)
        r_init = np.zeros(np.shape(H0)).astype(np.complex128)
        #r_init[15,15] = 1.0+0.0j
        
        for i in range(len(G)): #Initializing the density matrix considering a rot temperature of 4 K
            if i<16:
                r_init[i,i] = 1.0#/len(G)
            else:
                r_init[i,i] = 0.628*1.0#/len(G)

        
        pops = []
        Hint_Z = None
        Hint_X = None
        test_factor = 70
        #print('Creating obes')
        start = time.perf_counter()
        my_obe_1 = obe(Z0_field + Z2_field,[G,E],H0,Hint_1,0.96*BR,test_factor,mode = obe_mode)
        
        print(f"Z Hamiltonian took {time.perf_counter()-start}s")
        start = time.perf_counter()
        my_obe_2 = obe(X0_field + X2_field,[G,E],H0,[Hint_2,Hint_3],0.96*BR,test_factor,mode = obe_mode)
        print(f"X Hamiltonian took {time.perf_counter()-start}s")

        method = 'RK45'
        for count in range(passes):
            start = time.time()
            ans = my_obe_1.solve(steps,r_init,
                                max_step_size = 0.1/Gamma,#max_step_size,
                                package = package,
                                method = method)
            pops.append(ans)
            rho = np.array(ans[-1]) #gives the solution at the end of the time
            r_init = rho.reshape(n,n)
            #print(f"Z solve took {time.time()-start} s.")
            
            
            r_diag = np.round(np.diag(r_init),3)
            total_pop = np.sum(r_diag)
            sorted_indices = np.argsort(r_diag)[::-1]
            print(f"Pass Z:{count}",end=",")
            for indx in range(6):
                print(f"r{sorted_indices[indx]} = {r_diag[sorted_indices[indx]]}",end = ",")
            print(f"total_pop = {total_pop}")

            start = time.time()
            ans = my_obe_2.solve(steps,r_init,
                                max_step_size = 0.1/Gamma,#1/z_x_factor*max_step_size,
                                package=package,
                                method = method)
            pops.append(ans)
            rho = np.array(ans[-1]) #gives the solution at the end of the time
            r_init = rho.reshape(n,n)
            #print(f"X solve took {time.time()-start} s.")
            r_diag = np.round(np.diag(r_init),3)
            total_pop = np.sum(r_diag)
            
            sorted_indices = np.argsort(r_diag)[::-1]
            print(f"Pass X:{count}",end=",")
            for indx in range(6):
                print(f"r{sorted_indices[indx]} = {r_diag[sorted_indices[indx]]}",end = ",")
            print(f"total_pop = {total_pop}")
            
    
        r_level2optimize=np.real(r_init[level_to_optimize,level_to_optimize])
        rgg = 0

        for temp_count in range(len(G)):
            if temp_count != level_to_optimize:
                rgg += np.real(r_init[temp_count,temp_count])
        
        
        for i in range(len(G)):
            print(f"\'r_{i}\': {np.round(r_init[i,i],5)}",end = ",")
        print("----------")
        
        return -r_level2optimize,pops #nevergrad minimizes
    except Exception as e:
        print(e)
        print(rabi_det_params,Omegas)
        print("Error encountered with this input. Default value of 1 returned.")
        return 1.0,1.0

In [24]:
#Scan
def draw(rabi_det_params:list,Ns:list,Omegas:tuple,passes=1): #rabi_det_params was a field earlier
    from obe import pulse_np
    Omega_z_0,Omega_z_2, z_x_factor = Omegas
    Omega_x_0,Omega_x_2 = z_x_factor*Omega_z_0,z_x_factor*Omega_z_2

    [Nz_0,Nz_2,Nx_0,Nx_2] = Ns

    P_z_0 = []
    P_z_2 = []
    P_x_0 = []
    P_x_2 = []

    for i in range(len(current_field)): #Converting between Nevergrad array and field rabis and detunings
        current_field[i].rabi = rabi_det_params[i][0]
        current_field[i].detuning = rabi_det_params[i][1]
        if i < Nz_0:
            P_z_0.append(rabi_det_params[i][0]**2)
        elif i >= Nz_0 and i < (Nz_0 + Nz_2):
            P_z_2.append(rabi_det_params[i][0]**2)
        elif i >= (Nz_0 + Nz_2) and i < (Nz_0 + Nz_2 + Nx_0):
            P_x_0.append(rabi_det_params[i][0]**2)
        else:
            P_x_2.append(rabi_det_params[i][0]**2)

    total_pow_z_0 = np.sum(P_z_0)
    total_pow_z_2 = np.sum(P_z_2)
    total_pow_x_0 = np.sum(P_x_0)
    total_pow_x_2 = np.sum(P_x_2)


    P_z_0 = np.array(P_z_0)/np.sum(P_z_0)
    P_z_2 = np.array(P_z_2)/np.sum(P_z_2)
    P_x_0 = np.array(P_x_0)/np.sum(P_x_0)
    P_x_2 = np.array(P_x_2)/np.sum(P_x_2)


    threshold = 0.01 #rise time of the pulse signal used
    P_z_0[P_z_0 < threshold] = 0
    P_z_2[P_z_2 < threshold] = 0
    P_x_0[P_x_0 < threshold] = 0
    P_x_2[P_x_2 < threshold] = 0

    total_pow_z_0 = np.sum(P_z_0)
    total_pow_z_2 = np.sum(P_z_2)
    total_pow_x_0 = np.sum(P_x_0)
    total_pow_x_2 = np.sum(P_x_2)        

    P_z_0 = np.array(P_z_0)/total_pow_z_0*4*tsigma
    P_z_2 = np.array(P_z_2)/total_pow_z_2*4*tsigma
    P_x_0 = np.array(P_x_0)/total_pow_x_0*4*tsigma
    P_x_2 = np.array(P_x_2)/total_pow_x_2*4*tsigma        

    pos_z_0 = np.round(get_pos(P_z_0),5)
    pos_z_2 = np.round(get_pos(P_z_2),5)
    pos_x_0 = np.round(get_pos(P_x_0),5)
    pos_x_2 = np.round(get_pos(P_x_2),5)


    field = current_field
    field_Z_0 = field[0:
                        Nz_0]
    field_Z_2 = field[Nz_0:
                            Nz_0+Nz_2]
    field_X_0 = field[Nz_0+Nz_2:
                                Nz_0+Nz_2+Nx_0]
    field_X_2 = field[Nz_0+Nz_2+Nx_0:
                                    Nz_0+Nz_2+Nx_0+Nx_2]


    t=np.linspace(-0.1,8.4,10000)
    #Modify fields for N = 0 
    ############################
    plt.subplot(221)
    for i in range(len(field_Z_0)):
        field_Z_0[i].position = pos_z_0[i]
        field_Z_0[i].diameter = P_z_0[i]
        field_Z_0[i].rabi = Omega_z_0
        pulse_ = pulse_np(t,field_Z_0[i].position,field_Z_0[i].diameter)
        plt.plot(t,pulse_)
        max_idx = np.argmax(pulse_)
        if np.amax(pulse_>0.5):
            plt.text(t[max_idx],1/2,f"{desired_Z[i]} -> {np.round(field_Z_2[i].detuning,1)}",rotation=90,fontsize = 12)
        plt.title("Z fields : N = 0")

    plt.subplot(222)
    for i in range(len(field_X_0)):
        field_X_0[i].position = pos_x_0[i]
        field_X_0[i].diameter = P_x_0[i]
        field_X_0[i].rabi = 1/np.sqrt(2)*Omega_x_0
        pulse_=pulse_np(t,field_X_0[i].position,field_X_0[i].diameter)
        plt.plot(t,pulse_)
        max_idx = np.argmax(pulse_)
        if np.amax(pulse_>0.5):
            plt.text(t[max_idx],1/2,f"{desired_X[i]} -> {np.round(field_X_0[i].detuning,1)}",rotation=90,fontsize = 12)
        plt.title("X fields : N = 0")

    #Modify fields for N = 2
    ######################### 
    plt.subplot(223)
    for i in range(len(field_Z_2)):
        field_Z_2[i].position = pos_z_2[i]
        field_Z_2[i].diameter = P_z_2[i]
        field_Z_2[i].rabi = Omega_z_2
        pulse_ = pulse_np(t,field_Z_2[i].position,field_Z_2[i].diameter)
        plt.plot(t,pulse_)
        max_idx = np.argmax(pulse_)

        #if np.amax(pulse_>0.5):
        #    plt.text(t[max_idx],pulse_[max_idx]/2,f"{desired_Z[Nz_0+i]} -> {np.round(field_Z_2[i].detuning,1)}",rotation=90,fontsize = 6)
        plt.title("Z fields : N = 2")

    
    plt.subplot(224)
    for i in range(len(field_X_2)):
        field_X_2[i].position = pos_x_2[i]
        field_X_2[i].diameter = P_x_2[i]
        field_X_2[i].rabi = 1/np.sqrt(2)*Omega_x_2
        plt.plot(t,pulse_np(t,field_X_2[i].position,field_X_2[i].diameter))
        plt.title("X fields : N = 2")
    
    
    return 0



import nevergrad as ng

rabi_limit = 10
detuning_limit = 30
omegas_limit = 30
z_x_factor_limit = 1.0

passes = 9


# Define the parameter space
x_rabi = ng.p.Array(shape=(len(current_field),)).set_bounds(lower= 0, upper = 1)  # x is an array of size 2

y_det = ng.p.Array(shape=(len(current_field),)).set_bounds(lower = -1, upper = 1)    # y is an array of size 3
z_omega = ng.p.Tuple(
    ng.p.Scalar().set_bounds(lower = 0, upper = 1),           # z[0] is a scalar
    ng.p.Scalar().set_bounds(lower = 0, upper = 1),              # z[1] is a scalar
    ng.p.Scalar().set_bounds(lower = 0, upper = 1),           # factor between z and x
    
)

# Combine x, y, and z into a single parameter space
param_space = ng.p.Instrumentation(x_rabi, y_det, z_omega)
optimizer = ng.optimizers.DiagonalCMA(parametrization=param_space, budget=5000)

n_jobs =1#optimizer.num_workers

arr_loss = []
fname = f"./Bz_variation/XZ_rabi_det_N0_15_Bz_{Bz}_tsigma_{tsigma}_passes_{passes}_evolution_mac_CMA.csv"


with open(fname,'w') as file2write:
    file2write.write("Pop, (rabi, det) for the fields\n")
    file2write.write("transitions_Z :"+str(transitions_Z)+"\n")
    file2write.write("transitions_X :"+str(transitions_X)+"\n")
    optimizer.enable_pickling()
    for i in range(optimizer.budget//n_jobs):
        
        candidate_list = []
        #candidate_list = [optimizer.ask() for _ in range(n_jobs)]
        for counter in range(n_jobs):
            candidate_list.append(optimizer.ask())
        
        
        rabi_det_list = []
        omegas_list = []
    
    
        for candidate in candidate_list:
            rabis = candidate.args[0]
            rabis = np.maximum(rabis, 1e-5)
            detunings =  candidate.args[1]
            omegas = candidate.args[2]
            omegas_temp = [val*omegas_limit+1e-5 for val in omegas]
            omegas_temp[-1] = (omegas_temp[-1]+1e-5)*(z_x_factor_limit)/omegas_limit
            omegas = tuple(omegas_temp)
            

            rabi_det_array = [(rabis[j]*rabi_limit,detunings[j]*detuning_limit) for j in range(len(rabis))]
            rabi_det_list.append(rabi_det_array)
            omegas_list.append(omegas)
            print(omegas_list)
        
        #loss_list = Parallel(n_jobs=n_jobs)(delayed(r22)(rabi_det_list[k],Ns,omegas_list[k],passes=passes) for k in range(n_jobs))
        
        loss_list=[r22(rabi_det_list[k],Ns,omegas_list[k],passes=passes) for k in range(1)]
        

        
        for ii,(candidate,loss) in enumerate(zip(candidate_list,loss_list)):
            print(datetime.datetime.now(),ii,loss)
            optimizer.tell(candidate, loss)

            array_to_write = copy.deepcopy(rabi_det_list[ii])
            array_to_write.insert(0,loss)
            array_to_write.append(omegas_list[ii])
            
            file2write.write(str(array_to_write)+'\n')
            print("------------------------------------------------------------------------------")
            arr_loss.append(loss)
            plt.plot(arr_loss,'o-r')
            plt.pause(0.001)

# Retrieve the best solution found
recommendation = optimizer.provide_recommendation()
best_params = recommendation.value

print("Best Parameters Found:", best_params)


In [25]:
f= open('./Bz_variation/XZ_rabi_det_N0_15_Bz_20.0_tsigma_2.048_passes_9_evolution_mac_CMA.csv')
all_array = []
for i,line in enumerate(f):
    if i < 3:
        continue
    mylist = eval(line)
    all_array.append(mylist)
sorted_array = sorted(all_array, key=lambda item: item[0],reverse=True)
pop = [item[0] for item in all_array]
plt.plot(pop,'.-')

In [26]:
#arr = [-0.06794000930252098, (0.09073297687597676, -2.273033490125983), (1.422914889894515, -3.12866683381516), (2.0201918004489525, -0.6466940623066644), (2.1820373913369875, -2.9922987504997343), (1.587582684083073, -4.779832681581274), (0.6052025580529146, -3.517216140162518), (1.2085875845512954, -0.17035334723423454), (0.9253837313668611, 4.23577247140647), (0.18855871348290562, -3.745067422313548), (0.7667336526367492, 2.4519138309457897), (0.44046729243779886, -4.007941631942406), (0.6923217385593049, 2.1064145374621304), (1.157971473291706, 4.964374963730084), (3.6145493886817386, 1.067203235721742), (0.03799692778684676, -2.9329426680272492), (1.5723658910864438, 7.024505688988581), (2.501509350604561, 0.011040121333689807), (1.466832038489734, 4.4711490542032495), (0.5055610791485117, -1.7000413037717126), (0.786387301998399, 6.008626290135908), (0.7783688348496669, -0.18681766384374063), (0.1644155589080084, -2.5621698513401947), (1.1005815035768431, -1.1010740093490978), (2.6966079851669895, 5.604624213313732), (0.96729106232432, 1.880691471433173), (1.2230703136648238, 1.061196579401179), (0.8433524051571928, 1.5358719243840049), (1.7129094395007918, -2.7241441042414407), (0.2555747079634495, -7.584169170567547), (0.7080320268989198, -3.814646461724501), (0.021049955228157773, 3.0071931121467976), (0.7289276737688446, 4.331022099965598), (1.5754192872417363, 5.939603761046079), (1.0805057532620725, 1.476163129304841), (1.6130102892222562, 1.6982890050560062), (1.716301269606432, -3.5794192035559504), (1.1968250088264374, -13.27066667996463), (0.6258671068255195, -6.449632068938049), (0.20310026270297954, -3.6187879126852307), (1.079521368752466, 4.797067465917514), (0.37959928248407737, 3.6600429784243365), (1.0156854528201147, -2.8044991202557874), (0.18428892242321293, 1.239280928415368), (0.8626171380016505, 5.577215867772588), (1.8671877176569764, 3.3007740557658543), (0.5874596658687915, 0.5473610719002729), (1.9383358199736023, -4.8097944991982535), (0.44313004058940086, -2.79853156630245), (0.5173237744041382, -1.083970138392949), (1.9045831368846948, -0.7823111511314355), (0.5287378968884566, 9.277569784291478), (0.15192698778658814, 5.270893339227397), (1.2127669725059222, -4.5465833924340044), (0.3916955109355289, 2.3230348380945007), (1.6202377988803642, 2.201290536184822), (0.29409767990700714, 1.4589854589951186), (1.100570657806805, -4.758702895653554), (1.7257902239770009, 3.5885204594583566), (3.6214254479238797, 2.9591230936263613), (1.0040354944342094, 2.1895427790651123), (0.8368513898627776, -2.9991221819433003), (1.6681547772365808, 7.2959906562130055), (1.651133246753962, -2.7111256108972794), (1.4500543372781218, 8.397035852663201), (1.5405620460236067, -1.4437773315076767), (1.995982054819311, 3.044770070497677), (1.889802196090565, 2.020872323247071), (0.8712155212268656, -0.616539134102678), (1.182849190751816, -6.715062731068209), (2.5939025213561777, -6.828849662101289), (2.165158933707325, -4.035880113335514), (1.1841036985068665, -1.930072274588714), (0.5801509360462682, 12.19744255052058), (1.749330678795475, 1.9506920987692742), (1.1949129980521123, 1.7462597142725136), (0.4125718936599452, -1.1769335248565), (1.299215981762497, 5.168358941064932), (1.837642784289406, -3.1864500936782356), (0.1424274191293489, 0.5628539858855002), (0.05654506282804623, 4.916248208326636), (0.7017183759794897, -9.119612483641522), (1.4571869752703261, -7.958885712251331), (0.9624182118921467, 1.9899391967300364), (2.082639066153985, 4.327997118585684), (1.26514115544736, -6.927256727120644), (0.5140016162614581, -6.4987516002927075), (0.1169482415255914, -3.427022015801195), (1.4434114992926974, 2.9811537463436886), (1.0836016193325961, 8.699448424038888), (0.960379260378204, -0.6176088872493447), (0.9088143188502389, 0.3772018897158157), (0.17803984004142398, -1.5117189910711386), (0.9277672155422806, 3.381023825944648), (0.10391071835926706, 5.270674255277814), (0.5293269552070505, -4.897732211284179), (3.658095047561858, -4.360929422070814), (1.8543177031677174, -0.9423959424919761), (2.3625448676790066, 1.3571609058281724), (0.8534164556680625, 2.2693528144716426), (0.5476905916017321, 2.090545204630735), (0.08002587311796305, 3.0085035922949137), (1.6887626277317493, 2.157590486865952), (0.3695484782896069, 0.48075280187015934), (0.3860374707688181, 3.327822525346557), (1.170135526213319, 2.918630793761639), (0.8644522311173631, 3.967095458194705), (1.3346404479465226, 2.0093590031688224), (0.8103473753645314, -4.836158160619684), (0.686776452554268, 0.7629274791618968), (1.6598068764387812, 6.5690704642271065), (3.504701664862317, 6.101081296646262), (1.0118884061445192, -6.124059619188825), (0.38549992558738944, -2.3333669421939076), (1.3911317172495006, 2.6022372687416193), (0.039651077536118295, 6.734460039035152), (0.9714317181892316, -10.346241034124898), (1.5291804004229068, -2.3069034245156264), (0.43239293831356784, -4.576325782342108), (0.12134484221004313, 10.860386873450873), (0.23523963457434158, -0.6836553383172294), (2.572685921993911, 7.387747542657661), (0.12945341583016867, 1.204139971256047), (3.447524901106365, -3.269693195488022), (0.5238400843938644, -2.4299973494887888), (1.6177651165049858, -2.9928395040378204), (1.128068286662358, -5.396516148845301), (0.9125779386588752, -2.056969730726187), (1.3102320306168624, -5.064008556024956), (1.005008877691171, 4.66537772326928), (0.3632339764452425, 2.141292985932944), (0.27731274419885976, -4.620006899480542), (2.021876698853699, -0.6380410351092731), (0.12428762445106821, 0.3196023125901843), (1.4036199930401128, -0.41079117983711216), (2.4050392324606347, -0.9330085938454986), (0.5353006539816636, 8.96885778456409), (1.9042069624565077, -6.834272883203064), (2.479407379465764, -3.197668698584803), (1.6668828978328332, -4.464977701415675), (1.9449955478707683, 1.7803203407210442), (2.4372241357823805, 2.5327574172240306), (1.3242356518564198, 4.993397753045883), (0.45734536120831193, 0.2937699901250704), (1.609471283602035, -3.662048556261028), (0.04521884545398842, 1.1957442866640522), (0.45433123944100345, -3.209225994061415), (1.5060702922441913, -6.796723657102415), (1.3811647340544533, 5.690825531282342), (2.7769093169885517, -3.9368682724935224), (0.5625021059124233, 3.1255153141935765), (0.2245461228891258, 0.3433261516471288), (0.6356911109740375, 2.0664157204736044), (0.7105826443300073, -4.268755577537629), (2.770206311040624, 3.663550206161267), (0.5144826856747674, 0.27209905181819166), (0.03893034731183181, -0.04767976184365003), (0.9591837991601687, -1.6841584826756024), (0.7929870683494654, 1.2843183273769496), (0.04631892239003826, 3.101559739327982), (0.6347117224095844, -2.8801811572514), (2.2422695319272314, -1.079960553981822), (3.149582383033587, -3.9741759388970332), (0.9220774165990389, -4.5009779878111065), (4.2044740237040505, 3.664748563324358, 3.0902152706874677, 4.824891064120179, 0.5821615732054538)]
arr = sorted_array[-1]
print(arr)
print(arr[-1])

[np.float64(-35.21478098953151), (np.float64(3.195300375504177), np.float64(-8.752301830069891)), (np.float64(7.250432839097421), np.float64(13.868255695738478)), (np.float64(1.6270767101673878), np.float64(2.2714971134392568)), (np.float64(6.710557237097325), np.float64(1.5545285854562407)), (np.float64(8.815335852782216), np.float64(13.1785207815598)), (np.float64(3.4606234575083654), np.float64(9.799491589903626)), (np.float64(1.528230143333523), np.float64(0.4357321694759169)), (np.float64(4.646571264607843), np.float64(-5.252687413269311)), (np.float64(2.007857469623084), np.float64(0.7661099570188927)), (np.float64(9.354746930127048), np.float64(-6.76505647087024)), (np.float64(0.5088235141675782), np.float64(22.86498493837954)), (np.float64(9.116080761111608), np.float64(-21.096700926330954)), (np.float64(8.413280010432993), np.float64(-24.49349558386912)), (np.float64(4.9647109521824975), np.float64(2.0276216394794395)), (np.float64(1.1702684084738828), np.float64(-8.6486768443

: 

In [27]:
#arr = sorted_array[-1]
rabi_det_array = arr[1:-1]
omegas = arr[-1]
print(omegas)
omegas_list = []
for i,item in enumerate(omegas):
    if i ==1:
        omegas_list.append(0.75*item)
    elif i == 2:
        omegas_list.append(1.*item)    
    else:
        omegas_list.append(0.75*item)
print(omegas_list)
#draw(rabi_det_array,Ns,tuple(omegas_list),passes=9)
val,pops = r22(rabi_det_array,Ns,tuple(omegas_list),passes=24)

(9.56637695183178, 28.43773786915947, 0.9357248781233761)
[7.174782713873835, 21.328303401869604, 0.9357248781233761]
Hamiltonian construction : 0.6383549580350518
First Lambdify took 2.4977 s
Second Lambdify took 2.1829 s
Z Hamiltonian took 5.325448332936503s
Hamiltonian construction : 2.966475791996345
First Lambdify took 2.9965 s
Second Lambdify took 5.5559 s
X Hamiltonian took 11.52729229198303s
Solving started.
ODE solver took : 9.644830208970234 s
Time spent on lambdified Hinit = 1.5593811314320192s
Time spent on commutator numba = 1.7133804480545223s
Time spent on decay = 0.24145719059742987s
Time spent on repopulation = 0.09572155412752181s
Pass Z:0,r15 = (4.316+0j),r9 = (3.397+0j),r0 = (2.281+0j),r6 = (2.201+0j),r2 = (2.156+0j),r7 = (2.098+0j),total_pop = (64.399+0j)
Solving started.
ODE solver took : 11.041369750048034 s
Time spent on lambdified Hinit = 2.266995518701151s
Time spent on commutator numba = 1.6525970315560699s
Time spent on decay = 0.24536362697836012s
Time spen

In [ ]:
error

NameError: name 'error' is not defined

In [ ]:
1*16+0.628*80

66.24000000000001

In [ ]:
## read from json
import json
with open('./Bz_variation/XZ_rabi_det_N0_15_Bz_5.0_tsigma_2.048_passes_9_evolution_office_CMA.json','r') as f:
    data = json.load(f)
m = len(data['candidates'])
mylist = [val['loss'] for val in data['candidates'] ]
plt.plot(mylist,'-o')


In [ ]:
data["candidates"][0]['loss']

In [ ]:
G[-2].GtoJ()

(0.01168-0j) |N = 2, J = 1.5, F1 = 3.0, F = 3.5, mF = 3.5> + 
(0.01359+0j) |N = 2, J = 2.5, F1 = 3.0, F = 3.5, mF = 3.5> + 
(-0.00843+0j) |N = 2, J = 2.5, F1 = 4.0, F = 3.5, mF = 3.5> + 
(-0.9998+0j) |N = 2, J = 2.5, F1 = 4.0, F = 4.5, mF = 3.5>

In [ ]:
E[-1]

In [ ]:
import cProfile
import pstats

with cProfile.Profile() as pr:
    r22(rabi_det_array,Ns,omegas,passes=1)

stats = pstats.Stats(pr)
stats.sort_stats("tottime").print_stats(10)

cum_det : 0.008033985184738412
Lambdify took 6.6350 s
Z Hamiltonian took 11.199121083991486s
cum_det : 0.025400569778867066
Lambdify took 5.5822 s
X Hamiltonian took 14.05575879200478s
Z solve took 5.040600776672363 s.
Pass Z:0,r15 = (7.48+0j),r10 = (6.162+0j),r6 = (3.87+0j),r7 = (3.557+0j),r8 = (3.341+0j),r9 = (2.494+0j),total_pop = (62.63300000000001+0j)
X solve took 5.1989898681640625 s.
Pass X:0,r15 = (11.865+0j),r14 = (4.084+0j),r13 = (3.235+0j),r12 = (3.19+0j),r11 = (3.016+0j),r5 = (1.796+0j),total_pop = (58.164+0j)
'r_0': (1.32326+0j),'r_1': (0.56026+0j),'r_2': (0.64917+0j),'r_3': (0.85451+0j),'r_4': (0.05301+0j),'r_5': (1.7956+0j),'r_6': (0.32932+0j),'r_7': (0.58849+0j),'r_8': (1.09659+0j),'r_9': (0.54214+0j),'r_10': (1.12781+0j),'r_11': (3.01603+0j),'r_12': (3.19049+0j),'r_13': (3.23463+0j),'r_14': (4.0836+0j),'r_15': (11.86475+0j),'r_16': (0.48745+0j),'r_17': (0.29582+0j),'r_18': (0.22753+0j),'r_19': (0.21015+0j),'r_20': (0.20266+0j),'r_21': (0.18242+0j),'r_22': (0.18621+0j),

## Plot the population evolution

In [ ]:
levels = 112
passes = 9
# Generate 112 colors from a continuous colormap
cmap = plt.get_cmap('hsv')  # Try 'nipy_spectral', 'turbo', or 'gist_rainbow' too
colors = [cmap(i / levels) for i in range(levels)]
import random
random.seed(110)
colors = [(random.random(), random.random(), random.random()) for _ in range(112)]
index = np.arange(levels)*(levels+1)
t_block = 8.192
x = np.arange(5000)*t_block/5000
for i in range(2*passes):
    M = pops[i]
    N = np.real(M[:,index])
    for j in range(levels):
        if i == 0:
            plt.plot(x,N[:,j],color = colors[j],label = str(j));
        elif i == 2*passes-1:
            plt.plot(x,N[:,j],color = colors[j]);
            plt.text(x[-1]+1,N[-1,j],str(j),fontsize = 7)
        else:
            plt.plot(x,N[:,j],color = colors[j]);
    x += t_block 
#plt.legend();


# Generate 112 colors from a continuous colormap
levels = 112
cmap = plt.get_cmap('hsv')  # Try 'nipy_spectral', 'turbo', or 'gist_rainbow' too
colors = [cmap(i / levels) for i in range(levels)]
A_plot = []
passes = 9
for i in range(passes*2):
    A = pops[i]

    A_diag = [np.diag(item.reshape((levels,levels))) for item in A]
    A_plot.append(A_diag)

t=np.linspace(0,2*9*8.192,5000)
for i in range(levels):
    B_plot = 


x = [np.linspace(8.192*i,8.192*(i+1),5000) for i in range(2*passes)]
for i in range(passes*2):
    plt.plot(x[i],A_plot[i],color = colors);


In [ ]:
A_diag[0].shape

In [ ]:
error here

In [ ]:
ground_mF = [item.states[0].mF+np.linspace(-1/4,1/4,3) for item in G[0:16]]
ground_Energy = np.array([GH[i,i] for i in range(len(ground_mF))])
ground_Energy -= np.mean(ground_Energy)
min_ground_energy = np.amin(ground_Energy)
max_ground_energy = np.amax(ground_Energy)

excited_mF = [item.states[0].mF+np.linspace(-1/4,1/4,3) for item in E]
excited_Energy = np.array([EH[i,i] for i in range(len(E))])
excited_Energy -= np.mean(excited_Energy) - 1*np.abs(max_ground_energy-min_ground_energy)

"""
rabi_det_params = max_array # --------the solution from the optimization-----------------
for i in range(len(current_field)):
            current_field[i].rabi = rabi_det_params[i][0]
            current_field[i].detuning = rabi_det_params[i][1]
"""            
#Plot the energy levels
for i in range(len(ground_mF)):
    plt.plot(ground_mF[i],ground_Energy[i]+np.zeros(3))
    plt.text(ground_mF[i][1],ground_Energy[i]-8,f"G[{i}]",fontsize = 8)
for j in range(len(E)):
    plt.plot(excited_mF[j],excited_Energy[j]+np.zeros(3))
    plt.text(excited_mF[j][1],excited_Energy[j]-8,f"E[{j}]",fontsize = 8)
"""
rabis = []
for i in range(0,len(max_array),2):
    rabis.append(min_array[i])
max_rabi = np.amax(np.abs(np.array(rabis)))
    
# Plot the z transitions
for idx,item in enumerate(transitions_Z):
    i,j = item
    offset = random.uniform(-0.25,0.25)
    x = [ground_mF[i][1]+offset,excited_mF[j][1]+offset]
    y = [ground_Energy[i],excited_Energy[j]+current_field[idx].detuning]
    plt.plot(x,y,alpha = np.abs(current_field[idx].rabi/max_rabi),linewidth = np.abs(current_field[idx].rabi/max_rabi*5))


#Plot sigma transitions
for idx,item in enumerate(transitions_sigma):
    i,j = item
    offset = random.uniform(-0.05,0.05)
    x = [ground_mF[i][1]+offset,excited_mF[j][1]+offset]
    y = [ground_Energy[i],excited_Energy[j]+current_field[idx+N_z].detuning]
    #plt.plot(x,y,alpha = np.abs(current_field[idx+N_z].rabi/max_rabi),linewidth = np.abs(current_field[idx+N_z].rabi/max_rabi*10))
"""

'\nrabis = []\nfor i in range(0,len(max_array),2):\n    rabis.append(min_array[i])\nmax_rabi = np.amax(np.abs(np.array(rabis)))\n\n# Plot the z transitions\nfor idx,item in enumerate(transitions_Z):\n    i,j = item\n    offset = random.uniform(-0.25,0.25)\n    x = [ground_mF[i][1]+offset,excited_mF[j][1]+offset]\n    y = [ground_Energy[i],excited_Energy[j]+current_field[idx].detuning]\n    plt.plot(x,y,alpha = np.abs(current_field[idx].rabi/max_rabi),linewidth = np.abs(current_field[idx].rabi/max_rabi*5))\n\n\n#Plot sigma transitions\nfor idx,item in enumerate(transitions_sigma):\n    i,j = item\n    offset = random.uniform(-0.05,0.05)\n    x = [ground_mF[i][1]+offset,excited_mF[j][1]+offset]\n    y = [ground_Energy[i],excited_Energy[j]+current_field[idx+N_z].detuning]\n    #plt.plot(x,y,alpha = np.abs(current_field[idx+N_z].rabi/max_rabi),linewidth = np.abs(current_field[idx+N_z].rabi/max_rabi*10))\n'

In [ ]:
## Draw the sequences


In [ ]:
import joblib
print(joblib.cpu_count())

In [ ]:
G[85]

In [ ]:
1/96*0.3232

In [ ]:
A = {'r_0': (0.28453+0j),'r_1': (0.338+0j),'r_2': (0.24792+0j),'r_3': (0.20689+0j),'r_4': (0.29214+0j),'r_5': (0.12848+0j),'r_6': (0.01472+0j),'r_7': (0.03378+0j),'r_8': (0.04467+0j),'r_9': (0.02292+0j),'r_10': (0.59733+0j),'r_11': (0.2894+0j),'r_12': (0.21854+0j),'r_13': (0.26868+0j),'r_14': (0.40887+0j),'r_15': (34.25577+0j),'r_16': (0.08117+0j),'r_17': (0.09051+0j),'r_18': (0.05862+0j),'r_19': (0.05138+0j),'r_20': (0.03384+0j),'r_21': (0.06142+0j),'r_22': (0.04078+0j),'r_23': (0.05036+0j),'r_24': (0.06112+0j),'r_25': (0.03585+0j),'r_26': (0.03465+0j),'r_27': (0.03842+0j),'r_28': (0.03832+0j),'r_29': (0.0206+0j),'r_30': (0.03545+0j),'r_31': (0.07315+0j),'r_32': (0.03085+0j),'r_33': (0.04019+0j),'r_34': (0.05845+0j),'r_35': (0.04151+0j),'r_36': (0.05498+0j),'r_37': (0.04706+0j),'r_38': (0.03765+0j),'r_39': (0.01864+0j),'r_40': (0.2017+0j),'r_41': (0.05699+0j),'r_42': (0.05597+0j),'r_43': (0.02436+0j),'r_44': (0.01746+0j),'r_45': (0.04673+0j),'r_46': (0.11866+0j),'r_47': (0.16823+0j),'r_48': (0.13518+0j),'r_49': (0.0947+0j),'r_50': (0.17283+0j),'r_51': (0.13914+0j),'r_52': (0.11815+0j),'r_53': (0.07123+0j),'r_54': (0.02849+0j),'r_55': (0.04531+0j),'r_56': (0.04988+0j),'r_57': (0.04317+0j),'r_58': (0.03138+0j),'r_59': (0.06559+0j),'r_60': (0.03857+0j),'r_61': (0.20424+0j),'r_62': (0.04278+0j),'r_63': (0.20281+0j),'r_64': (0.00866+0j),'r_65': (0.01963+0j),'r_66': (0.01569+0j),'r_67': (0.01373+0j),'r_68': (0.01887+0j),'r_69': (0.02796+0j),'r_70': (0.02123+0j),'r_71': (0.02287+0j),'r_72': (0.00944+0j),'r_73': (0.03005+0j),'r_74': (0.00965+0j),'r_75': (0.01694+0j),'r_76': (0.0245+0j),'r_77': (0.01893+0j),'r_78': (0.03862+0j),'r_79': (0.01779+0j),'r_80': (0.01156+0j),'r_81': (0.00875+0j),'r_82': (0.628+0j),'r_83': (0.07616+0j),'r_84': (0.34115+0j),'r_85': (0.00563+0j),'r_86': (0.43145+0j),'r_87': (0.02079+0j),'r_88': (0.35117+0j),'r_89': (0.0693+0j),'r_90': (0.49956+0j),'r_91': (0.40179+0j),'r_92': (0.39419+0j),'r_93': (0.50787+0j),'r_94': (0.63095+0j),'r_95': (0.628+0j)}

In [ ]:
plt.figure()
pop_init = 16+80*0.628
sum_val = 0
for i in range(96):
    plt.plot(i,np.real(A[f"r_{i}"]),'o')
    sum_val += A[f"r_{i}"]
x=[0,96]
y = [0.628,0.628]
z = [1,1]
plt.plot(x,y,'r')
plt.plot(x,z,'k')
plt.xlabel('Ground state index',fontsize = 14)
plt.ylabel('Gain ',fontsize = 14)
plt.xticks(fontsize = 14)
plt.yticks(fontsize = 14)
print(f"Init pop = {np.round(pop_init,3)}, final pop = {np.round(sum_val,3)}, lost pop = {np.round(pop_init-sum_val,3)}")


Init pop = 66.24, final pop = (46.282+0j), lost pop = (19.958+0j)


In [ ]:
import sympy
print(sympy.__version__)
